# LoRA-corrected teacher eval: HumanEval-TS (full 159-problem set)

Part of plan.md step 3.3 -- the full-scale follow-up to
`11a_teacher_corrected_lora_humaneval_ts_sample.ipynb`'s n=40 diagnostic.
Same suite and harness as `10a_teacher_corrected_humaneval_ts.ipynb` (the
full-parameter corrected teacher's run), same export-stripping fix
(`bonsai.eval.register`) -- only the model pointer differs, to the
LoRA-corrected teacher (`sartajbhuvaji/bonsai-qwen3-30b-a3b-teacher-corrected-lora`, 3.3), trained on the
postprocessed retraining pool (3.4: `node_modules`/`.d.ts` excluded,
samples over 2048 tokens dropped rather than truncated).

Comparison point: base model 0.6708, full-parameter corrected teacher
0.5660 (both full 159-problem set). Reuses `batch_size=8` (not the
harness's namesake-suggesting-bigger default) -- `batch_size=32` hung for
over an hour during the original 1.2 baseline eval; this MoE model's
batched decode doesn't amortize across sequences that route to different
experts the way a dense model's would.

Needs a rented GPU (1x 80GB) with the `eval` extra installed.

In [1]:
import json

from bonsai.config import load_config
from bonsai.tracking import new_run, save_config_snapshot

config = load_config("../configs/qwen3-30b-a3b.toml")
model_repo_id = "sartajbhuvaji/bonsai-qwen3-30b-a3b-teacher-corrected-lora"
run_dir = new_run("teacher-corrected-lora-eval")
save_config_snapshot(run_dir, config)

print(f"model: {model_repo_id}")
print(f"run dir: {run_dir}")

model: sartajbhuvaji/bonsai-qwen3-30b-a3b-teacher-corrected-lora
run dir: runs/20260922T233140Z-570fad-teacher-corrected-lora-eval


In [2]:
import os

os.environ["MODEL_REPO_ID"] = model_repo_id
os.environ["MODEL_REVISION"] = "main"
os.environ["RUN_DIR"] = str(run_dir)

In [3]:
!stdbuf -o0 -e0 accelerate launch ../tools/bigcode_eval_main.py \
    --model "$MODEL_REPO_ID" --revision "$MODEL_REVISION" --precision bf16 \
    --tasks multiple-ts \
    --max_length_generation 512 --batch_size 8 \
    --allow_code_execution --save_generations \
    --metric_output_path "$RUN_DIR/humaneval-ts.json" \
    --save_generations_path "$RUN_DIR/humaneval-ts-generations.json" \
    2>&1 | tee /tmp/live_humaneval_ts_lora_full.log

assert _exit_code == 0, "accelerate launch failed for multiple-ts"  # noqa: F821 -- set by the ! shell magic above at runtime

The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_processes` was set to a value of `1`
	`--num_machines` was set to a value of `1`
	`--mixed_precision` was set to a value of `'no'`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.


Selected Tasks: ['multiple-ts']
Loading model in bf16
Loading weights:   0%|          | 0/531 [00:00<?, ?it/s]

Loading weights:   1%|          | 4/531 [00:00<00:53,  9.90it/s]

Loading weights:   1%|          | 5/531 [00:01<02:53,  3.03it/s]

Loading weights:   3%|▎         | 15/531 [00:01<00:44, 11.69it/s]

Loading weights:   3%|▎         | 18/531 [00:02<01:04,  7.99it/s]

... (409 outputs left out)

 99%|█████████▊| 157/159 [02:13<00:01,  1.16it/s]

 99%|█████████▉| 158/159 [02:14<00:00,  1.14it/s]

100%|██████████| 159/159 [02:15<00:00,  1.18it/s]


{
  "multiple-ts": {
    "pass@1": 0.5723270440251572
  },
  "config": {
    "prefix": "",
    "do_sample": true,
    "temperature": 0.2,
    "top_k": 0,
    "top_p": 0.95,
    "n_samples": 1,
    "eos": "<|endoftext|>",
    "seed": 0,
    "model": "sartajbhuvaji/bonsai-qwen3-30b-a3b-teacher-corrected-lora",
    "modeltype": "causal",
    "peft_model": null,
    "revision": "main",
    "use_auth_token": false,
    "trust_remote_code": false,
    "tasks": "multiple-ts",
    "instruction_tokens": null,
    "batch_size": 8,
    "max_length_generation": 512,
    "precision": "bf16",
    "load_in_8bit": false,
    "load_in_4bit": false,
    "left_padding": false,
    "limit": null,
    "limit_start": 0,
    "save_every_k_tasks": -1,
    "postprocess": true,
    "allow_code_execution": true,
    "generation_only": false,
    "load_generations_path": null,
    "load_data_path": null,
    "metric_output_path": "runs/20260922T233140Z-570fad-teacher-corrected-lora-eval/humaneval-ts.json",
    "s

In [4]:
humaneval_ts = json.loads((run_dir / "humaneval-ts.json").read_text())
print("humaneval-ts (LoRA-corrected teacher, full 159-problem set):", humaneval_ts)

humaneval-ts (LoRA-corrected teacher, full 159-problem set): {'multiple-ts': {'pass@1': 0.5723270440251572}, 'config': {'prefix': '', 'do_sample': True, 'temperature': 0.2, 'top_k': 0, 'top_p': 0.95, 'n_samples': 1, 'eos': '<|endoftext|>', 'seed': 0, 'model': 'sartajbhuvaji/bonsai-qwen3-30b-a3b-teacher-corrected-lora', 'modeltype': 'causal', 'peft_model': None, 'revision': 'main', 'use_auth_token': False, 'trust_remote_code': False, 'tasks': 'multiple-ts', 'instruction_tokens': None, 'batch_size': 8, 'max_length_generation': 512, 'precision': 'bf16', 'load_in_8bit': False, 'load_in_4bit': False, 'left_padding': False, 'limit': None, 'limit_start': 0, 'save_every_k_tasks': -1, 'postprocess': True, 'allow_code_execution': True, 'generation_only': False, 'load_generations_path': None, 'load_data_path': None, 'metric_output_path': 'runs/20260922T233140Z-570fad-teacher-corrected-lora-eval/humaneval-ts.json', 'save_generations': True, 'load_generations_intermediate_paths': None, 'save_genera

## Comparison

| | pass@1 |
|---|---|
| base model (unpruned, uncorrected) | 0.6708 |
| full-parameter corrected teacher | 0.5660 |
| LoRA-corrected teacher (this notebook) | see above |